# GenCare AI: Data generatie
---
Auteur:   Eva Rombouts  
Datum:    december 2023  
Script:   Dit script genereert fictieve zorgdata met behulp van de OpenAI API. 

---

## Setup

In dit script wordt gebruik gemaakt van de OpenAI bibliotheek. Deze biedt de API interface voor de verschillende AI-modellen ontwikkeld door OpenAI, zoals GPT-3 en GPT-4. Wij zullen het gebruiken voor text generation. 

Let op. Het instellen van een OpenAI API key is vereist voor het draaien van deze notebook. (https://platform.openai.com/docs/quickstart?context=python)

Het gebruik van OpenAI is niet gratis. Het genereren van 24 clientenprofielen met gpt-3.5-turbo kost ongeveer 3 cent. Met gpt-4 krijg je aanzienlijk betere modellen, dan betaal je 70 cent. 
Voor 10 weken rapportages van 24 clienten betaal je 70 cent met 3.5 turbo. 

De data wordt opgeslagen in dataframes. Hiervoor wordt uiteraard pandas geimporteerd. 

In [ ]:
from openai import OpenAI 
import pandas as pd
import re
import time

We gaan een afdeling vullen met clienten. Geef hieronder de afdelingsnaam aan en het aantal clienten dat hier ligt.  Voor elke client worden dagelijkse rapportages geschreven voor een x aantal weken, ook op te geven in de parameters hieronder. 

Voor tekstgeneratie moet het model worden opgegeven. En de seed wordt gebruikt om vergelijkbare (ze worden niet geheel identiek...) teksten te genereren.

Tot slot moeten de filenames worden opgeslagen voor het opslaan van de csv bestanden. Zorg ervoor dat de map bestaat. 

In [ ]:
afdelingsnaam = 'eikenboom'
aantal_clienten = 24
num_weeks = 10

seed = 6
model_profielen = 'gpt-4'
model_rapportages = 'gpt-3.5-turbo'

filename_profielen = f'../zorgdata/gci_clienten_{afdelingsnaam}.csv'
filename_rapportages = f'../zorgdata/gci_rapportages_{afdelingsnaam}.csv'

## Data generatie functie

Onderstaande functie produceert AI-gegenereerde data.

Allereerst wordt een instantie gemaakt van de OpenAI-client. Hiermee wordt verbinding gemaakt met de OpenAI API.  

Vervolgens genereert chat.completions.create() de tekst.  

Voorbeeld van de OpenAI documentatie:  
openai.chat.completions.create(  
model="gpt-3.5-turbo",  
  messages=[  
        {"role": "system", "content": "You are a helpful assistant."},  
        {"role": "user", "content": "Who won the world series in 2020?"},  
        {"role": "assistant", "content": "The Los Angeles Dodgers won the World Series in 2020."},  
        {"role": "user", "content": "Where was it played?"}  
    ]  
)

Argumenten:  
- s_role: De system rol bepaalt hoe de 'assistent' zich gedraagt. Je kan instructies meegeven over zijn doel en manier van antwoorden. De system rol wordt slechts één keer gedefinieerd.  
- u_role: De prompt van de 'user' waar de 'assistent' een antwoord (of een completion) op geeft. Eventueel kan dit de start zijn van een dialoog, maar dat gebruiken wij niet. 
- model: Voor mogelijke modellen zie: https://platform.openai.com/docs/models
- seed: spreekt voor zich
- n: Aantal completions die de functie teruggeeft. 

Het resultaat is een object met een lijst choices. De term choices wordt gebruikt voor de verschillende gegenereerde teksten. De inhoud van de tekst met index i wordt zo opgehaald: resultaat.choices[i].message.content

In [ ]:
def genereer_zorgdata(s_role, u_role, model='gpt-3.5-turbo', seed=None, n=1):
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": s_role},
                {"role": "user", "content": u_role}
            ],
            seed=seed,
            temperature=1.0,
            n=n
        )
        return completion
    except Exception as e:
        print(f"Er is een fout opgetreden: {e}")
        return None

## Genereer cliëntprofielen
Hieronder wordt de prompt gegeven.  
Hiervoor wordt dus apart gebruik gemaakt van een system role, die beschrijft hoe het model zich dient te gedragen.  
In de user prompt wordt de daadwerkelijke opdracht voor textgeneratie gevraagd. 

In [ ]:
s_role_profielen = '''
Je specialiseert je in het genereren van fictieve gegevens voor helpend en verzorgend personeel in verpleeghuizen, met een focus op cliënten met een psychogeriatrische aandoening. Jouw expertise omvat het creëren van realistische cliëntscenario's, cliëntendossiers en zorgplannen die specifiek zijn afgestemd op de behoeften van deze doelgroep. Deze gegevens moeten toegankelijk en relevant zijn voor personeel zonder uitgebreide medische kennis, en moeten belangrijke aspecten van de dagelijkse zorg en ondersteuning voor deze cliënten bevatten.
'''

u_role_profielen = '''
Schrijf een profiel van een cliënt die is opgenomen op een psychogeriatrische afdeling van het verpleeghuis. Gebruik onderstaande structuur:

Naam: [Meneer/Mevrouw Voornaam Achternaam]
Type dementie: [kies uit: Alzheimer, gemengde dementie, vasculaire dementie, lewy body dementie, parkinsondementie, FTD, hou rekening met hoe vaak deze vormen van dementie in de populatie voorkomen]
Lichamelijke klachten: [lichamelijke klachten]
Beschrijving cliënt: [een korte beschrijving van karakter en relevante biografische gegevens]
Belangrijkste zorgbehoefte:
- ADL: [Beschrijf ADL hulp]
- cognitie / probleemgedrag: [beschrijf voor de zorg relevante aspecten van cognitie en probleemgedrag. Varieer met de ernst van het probleemgedrag van rustige cliënten, gemiddeld onrustige clienten tot cliënten die fors apathisch, onrustig, angstig, geagiteerd of zelfs agressief kunnen zijn]

Vermijd veelvoorkomende namen als Jansen, de Vries en Fatima.
Vermijd stereotypen in beroepen en achtergronden. Niet elke vrouw was lerares en niet elke man was bankdirecteur of architect.
'''

Als de profielen al zijn gegenereerd, lees dan het bestand in (en sla de volgende cellen over)

In [ ]:
print(filename_profielen)
# df_profielen = pd.read_csv(filename_profielen)

In onderstaande code wordt de api daadwerkelijk aangeroepen voor het genereren van de tekst. 

In [ ]:
profielen = genereer_zorgdata(s_role=s_role_profielen, u_role=u_role_profielen, model=model_profielen, seed=seed, n=aantal_clienten)

Zet om in pandas df

In [ ]:
df_profielen = pd.DataFrame([{"profiel": choice.message.content} for choice in profielen.choices])

df_profielen is nu een dataframe met een enkele kolom. Namelijk de gegenereerde tekst.  
Hier maken we wat aanpassingen op: 
- We genereren een client_id in de vorm van een uniek kamernummer. 
- De naam wordt uit het profiel gehaald. Dit is de eerste regel van de tekst
- Het geslacht wordt ontleend aan Meneer/mevrouw
- De naam wordt gestript van meneer/mevrouw

In [ ]:
df_profielen = (df_profielen
    .assign(client_id=lambda x: x.index.map(lambda i: f'kamer{i+1:02d}'))
    .assign(naam=lambda x: x['profiel'].str.replace('Naam: ', '').str.split('\n').str[0])
    .assign(geslacht=lambda x: x['naam'].apply(lambda y: 'man' if 'Meneer' in y else ('vrouw' if 'Mevrouw' in y else 'onbekend')))
    .assign(naam=lambda x: x['naam'].str.replace('Meneer', '').str.replace('Mevrouw', '').str.strip())
    [['client_id', 'naam', 'geslacht', 'profiel']]
)

En sla op als csv file

In [ ]:
df_profielen.to_csv(filename_profielen, index=False)

## Genereer rapportages

Het genereren van de rapportages gaat vergelijkbaar, met enkele aanpassingen. De system role heb ik hetzelfde gehouden.  
De user prompt bevat nu een combinatie van de gegenereerde profielen en de prompt. Deze worden verderop aan elkaar geknoopt. 

In [ ]:
# Definieer inhoud rollen
s_role_rapportages = '''
Je specialiseert je in het genereren van fictieve gegevens voor helpend en verzorgend personeel in verpleeghuizen, met een focus op cliënten met een psychogeriatrische aandoening. Jouw expertise omvat het creëren van realistische cliëntscenario's, cliëntendossiers en zorgplannen die specifiek zijn afgestemd op de behoeften van deze doelgroep. Deze gegevens moeten toegankelijk en relevant zijn voor personeel zonder uitgebreide medische kennis, en moeten belangrijke aspecten van de dagelijkse zorg en ondersteuning voor deze cliënten bevatten.
'''

u_role_rapportages_instruction = '''
\n
Genereer fictieve zorgrapportages voor deze cliënt voor een periode van zeven dagen. 
Elke rapportage dient de volgende structuur te hebben: 
---StartRapportage---
Dag: [Dag van de week]
Niveau: [Helpende/Verzorgende/Verpleegkundige]
Onrustscore: [Onrustscore]
Rapportage: [Inhoud van de rapportage]
---EindRapportage---

De rapportages moeten afwisselend geschreven worden door een helpende (20% kans), een verzorgende (60% kans), of een verpleegkundige (20% kans). 
In de inhoud van elke rapportage komt meestal één, soms meer eigenschappen of beperkingen van de cliënt naar voren. Dit kunnen functionele of lichamelijke beperkingen zijn. En soms onrust. Wissel hiermee af, niet elke rapportage gaat over gedrag.

Geef elke rapportage een onrustscore: Een heel getal tussen 0 en 100, wat de mate weergeeft waarin de cliënt onrust vertoont in deze rapportage:
0-20: Geen onrust
21-40: Lichte onrust
41-60: Matige onrust
61-80: Ernstige onrust
81-100: Extreme onrust

Wees subtiel in de omschrijving van de beperkingen en/of de onrust. Noem het niet expliciet bij naam, maar laat de beperking doorschemeren in het functioneren van de client.

'''

En hieronder volgt weer het daadwerkelijk genereren van de rapportages. Hiervoor wordt dus over alle profielen geloopt, vervolgens wordt het profiel aan de instruction prompt geknoopt en er wordt een num_weeks aantal weekrapportages gegenereerd. De keuze om meerdere rapportages, een voor elke dag van de week, te vragen, is bewust genomen. Ik heb ook overwogen om individuele rapportages te maken, maar dit gaat minder gemakkelijk. Het model heeft dan de neiging om langere, ingewikkelder rapportages te genereren. Bovendien hoop ik dat door de teksten van meerdere chronologische rapportages te laten genereren, inherent duidelijk wordt dat het gaat om een client die over de tijd vaker wordt geobserveerd. Mogelijk zit er een soort lijn in de rapportages, dat bijvoorbeeld op woensdag wordt teruggekomen op iets wat maandag is gebeurd. Maar ik geloof niet dat hij dat doet...  
Het maakt het verderop wel weer iets ingewikkelder, want de weekrapportages moeten in dagrapportages worden gesplitst. 

In [ ]:
weekrapportage_list = []
for index, row in df_profielen.iterrows():
    start = time.time()
    client_id = row['client_id']
    print(client_id) # Print om voortgang bij te houden
    # Voeg de instructietekst toe aan het profiel
    u_role_rapportages = row['profiel'] + u_role_rapportages_instruction

    # Genereer de rapportages
    weekrapportages = genereer_zorgdata(s_role=s_role_rapportages, u_role=u_role_rapportages, model=model_rapportages, seed=seed, n=num_weeks)

    # Voeg elke rapportage toe aan de lijst
    for weekrapportage in weekrapportages.choices:
        weekrapportage_list.append({
            'client_id': client_id, 
            'weekno': weekrapportage.index,
            'rapportage': weekrapportage.message.content,
        })
    
    print (f"Verstreken tijd: {round(time.time()-start)} seconden")

We maken gebruik van de structuur van de rapportages om de verschillende rapportages te splitsen. 

In [ ]:
def split_rapportage(rapportage_tekst):
    # Combineer start- en eindpatronen in één patroon
    patroon = r'\s*-+\s*(startrapportage|eindrapportage)\s*-+\s*'
    splitpatroon = r'[\n\s]*§+[\n\s]*'

    # Vervang zowel start- als eindpatronen door §
    rapportage = re.sub(patroon, '§', rapportage_tekst, flags=re.IGNORECASE)

    # Verwijder § aan het begin en het einde (indien aanwezig)
    rapportage = re.sub(r'^' + splitpatroon, '', rapportage)
    rapportage = re.sub(splitpatroon + r'$', '', rapportage)

    # Voer een split uit
    rapportages = re.split(splitpatroon, rapportage)
    return rapportages


En we maken gebruik van de structuur om de gegevens van de rapportages te parsen

In [ ]:
def parse_dagrapportage(dagrapportage):
    # Zorg dat de onrustscore altijd op een nieuwe regel begint
    rapportage = re.sub(r'\n*(onrustscore)', '\nOnrustscore', dagrapportage, flags=re.IGNORECASE)
    # Zorg dat de rapportage direct achter de tekst 'Rapportage:' begint (dus zonder newline)
    rapportage = re.sub(r'\n*(rapportage:)[\n\s]*', '\nRapportage: ', rapportage, flags=re.IGNORECASE)

    rapportage_delen = re.split(r'\n', rapportage)

    parsed_data = {
        "dag": None,
        "niveau": None,
        "rapportage": None,
        "onrustscore": None
    }

    for deel in rapportage_delen:
        if deel.startswith('Dag:'):
            parsed_data["dag"] = deel[len('Dag:'):].strip().lower()
        elif deel.startswith('Niveau:'):
            parsed_data["niveau"] = deel[len('Niveau:'):].strip()
        # elif deel.startswith('Onrustscore:'):
        #     parsed_data["onrustscore"] = deel[len('Onrustscore:'):].strip()
        elif deel.startswith('Rapportage:'):
            parsed_data["rapportage"] = deel[len('Rapportage:'):].strip()
        elif deel.startswith('Onrustscore:'):
            # Zet de onrustscore om naar een integer
            score = deel[len('Onrustscore:'):].strip()
            try:
                parsed_data["onrustscore"] = int(score)
            except ValueError:
                # Handel eventuele conversiefouten af
                parsed_data["onrustscore"] = None

    return parsed_data

In [ ]:
client_rapportages = []

for weekrapportage in weekrapportage_list:
    dagrapportages = split_rapportage(weekrapportage['rapportage'])
    for dagrapportage in dagrapportages:
        parsed_dagrapportage = parse_dagrapportage(dagrapportage)
        parsed_dagrapportage['client_id'] = weekrapportage['client_id']
        parsed_dagrapportage['weekno'] = weekrapportage['weekno']+1 # Plus een om week 0 te voorkomen
        client_rapportages.append(parsed_dagrapportage)

In [ ]:
df_rapportages = pd.DataFrame(client_rapportages)

In [ ]:
df_rapportages = df_rapportages[['client_id', 'weekno', 'dag', 'niveau', 'rapportage', 'onrustscore']]
df_rapportages.head()

In [ ]:
df_rapportages.to_csv(filename_rapportages, index=False)